# MLflow quickstart — NHDW workshop

MLflow is already wired into this lab. This notebook runs the whole loop:
**track params, log a metric, and log a trained model** — with the model
artifact uploaded *through* the tracking server, so you need no S3 credentials.

1. `import nhdw_mlflow` connects to the in-cluster MLflow server and scopes
   runs to **your project's experiment** (`/projects/<your-slug>`).
2. We train a trivial scikit-learn model and log params + a metric + the model.
3. We print a clickable link to the run in the MLflow UI.

Open the MLflow UI at **https://mlflow.127.0.0.1.nip.io** to browse your runs.

In [ ]:
import nhdw_mlflow   # sets tracking URI + your project experiment (prints them)
import mlflow
import mlflow.sklearn

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# A tiny, self-contained demo frame — two feature columns, one target.
df = pd.DataFrame({
    "anc_visits": [120, 140, 90, 200, 160, 75, 180, 130],
    "facilities": [10, 12, 8, 18, 14, 6, 16, 11],
    "deliveries": [95, 110, 70, 165, 130, 55, 150, 100],
})
X = df[["anc_visits", "facilities"]]
y = df["deliveries"]
df

In [ ]:
with nhdw_mlflow.start_run("quickstart-linreg") as run:
    model = LinearRegression().fit(X, y)
    r2 = model.score(X, y)

    mlflow.log_param("features", "anc_visits,facilities")
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_metric("r2", float(r2))
    mlflow.sklearn.log_model(model, "model", input_example=X.head(2))

    print(f"R^2 = {r2:.3f}")

print("\nRun recorded — open it in the MLflow UI:")
print(nhdw_mlflow.run_url(run))

That's the full loop. In the MLflow UI you'll find the run under your
project's experiment, with the params, the `r2` metric, and a `model/`
artifact. Load it straight back from the run (downloads via the server too):

```python
loaded = mlflow.pyfunc.load_model(f"runs:/{run.info.run_id}/model")
loaded.predict(X.head(2))
```